In [ ]:
import subprocess

def setup_environment():
    subprocess.run(["apt-get", "-qq", "update"])
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"])
    subprocess.run(["pip", "install", "-q", "speechbrain==0.5.16", "pyannote.metrics", "datasets", "soundfile"])
    subprocess.run(["pip", "install", "-q", "git+https://github.com/m-bain/whisperx.git"])
    subprocess.run(["pip", "install", "-q", "torch==2.4.0", "torchvision==0.19.0", "torchaudio==2.4.0", "triton<3.2.0"])
    subprocess.run(["pip", "install", "-q", "-U", "vllm==0.6.1.post2", "numpy<2.0.0", "transformers==4.43.3"])

setup_environment()
print("Setup complete. Restart session required.")

In [ ]:
import subprocess
from pathlib import Path

AUDIO_DIR = Path("/content/evaluation_dataset/audio")
PROCESS_DIR = Path("/content/processed_mono")
PROCESS_DIR.mkdir(exist_ok=True)

for audio_file in AUDIO_DIR.glob("*.wav"):
    mono_path = PROCESS_DIR / audio_file.name
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(audio_file), "-ac", "1", "-ar", "16000", str(mono_path)],
        check=True,
        capture_output=True
    )
print("Audio processing complete.")

Preparing dataset audio for WhisperX...
All files converted to 16kHz Mono and ready for ASR.


In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
print("Authentication loaded.")

HF_TOKEN successfully loaded from Colab secrets.


In [ ]:
import os
from pathlib import Path
import soundfile as sf
from datasets import load_dataset

HF_TOKEN = os.environ.get("HF_TOKEN")
BASE_DIR = Path("/content/evaluation_dataset")
AUDIO_DIR = BASE_DIR / "audio"
GT_DIR = BASE_DIR / "ground_truth"
PRED_DIR = BASE_DIR / "predictions"

for d in [AUDIO_DIR, GT_DIR, PRED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

dataset = load_dataset("talkbank/callhome", "eng", split="data[:3]", token=HF_TOKEN)

for i, item in enumerate(dataset):
    file_id = f"callhome_eng_{i}"
    sf.write(str(AUDIO_DIR / f"{file_id}.wav"), item["audio"]["array"], item["audio"]["sampling_rate"])

    with open(GT_DIR / f"{file_id}.rttm", "w", encoding="utf-8") as f:
        for start, end, spk in zip(item["timestamps_start"], item["timestamps_end"], item["speakers"]):
            f.write(f"SPEAKER {file_id} 1 {start:.3f} {(end - start):.3f} <NA> <NA> {spk} <NA> <NA>\n")

print("Dataset initialized.")

eng/data-00000-of-00005.parquet:   0%|          | 0.00/446M [00:00<?, ?B/s]

eng/data-00001-of-00005.parquet:   0%|          | 0.00/488M [00:00<?, ?B/s]

eng/data-00002-of-00005.parquet:   0%|          | 0.00/473M [00:00<?, ?B/s]

eng/data-00003-of-00005.parquet:   0%|          | 0.00/438M [00:00<?, ?B/s]

eng/data-00004-of-00005.parquet:   0%|          | 0.00/453M [00:00<?, ?B/s]

Generating data split:   0%|          | 0/140 [00:00<?, ? examples/s]

Dataset structure initialized at /content/evaluation_dataset. Processed 3 samples.


In [ ]:
import os
import gc
import json
import torch
import whisperx
from whisperx.diarize import DiarizationPipeline
from pathlib import Path

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

def free_vram():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

asr_model = whisperx.load_model("large-v2", DEVICE, compute_type=COMPUTE_TYPE)
diarize_model = DiarizationPipeline(model_name="pyannote/speaker-diarization-3.1", device=DEVICE)

AUDIO_DIR = Path("/content/evaluation_dataset/audio")
PRED_DIR = Path("/content/evaluation_dataset/predictions")
JSON_DIR = Path("/content/")

for audio_path in AUDIO_DIR.glob("*.wav"):
    file_id = audio_path.stem
    audio = whisperx.load_audio(str(audio_path))

    result = asr_model.transcribe(audio, batch_size=16)
    align_model, metadata = whisperx.load_align_model(language_code=result["language"], device=DEVICE)
    result = whisperx.align(result["segments"], align_model, metadata, audio, DEVICE, return_char_alignments=False)

    del align_model
    free_vram()

    diarize_segments = diarize_model(audio)
    result = whisperx.assign_word_speakers(diarize_segments, result)

    first_speaker = next((seg.get("speaker") for seg in result["segments"] if seg.get("speaker", "UNKNOWN") != "UNKNOWN"), None)
    role_map = {}
    if first_speaker:
        role_map[first_speaker] = "Operator"
        for segment in result["segments"]:
            spk = segment.get("speaker", "UNKNOWN")
            if spk != "UNKNOWN" and spk != first_speaker:
                role_map[spk] = "Client"
                break

    with open(PRED_DIR / f"{file_id}.rttm", "w", encoding="utf-8") as f:
        for segment in result["segments"]:
            raw_spk = segment.get("speaker", "UNKNOWN")
            if raw_spk == "UNKNOWN": continue
            mapped_spk = role_map.get(raw_spk, raw_spk)
            f.write(f"SPEAKER {file_id} 1 {segment['start']:.3f} {(segment['end'] - segment['start']):.3f} <NA> <NA> {mapped_spk} <NA> <NA>\n")

    transcript_data = {
        "file_id": file_id,
        "segments": [
            {
                "speaker": role_map.get(seg.get("speaker", "UNKNOWN"), seg.get("speaker", "UNKNOWN")),
                "text": seg.get("text", "").strip(),
                "start": seg.get("start"),
                "end": seg.get("end")
            }
            for seg in result["segments"]
        ]
    }
    with open(JSON_DIR / f"{file_id}.json", "w", encoding="utf-8") as f:
        json.dump(transcript_data, f, ensure_ascii=False, indent=4)

free_vram()
print("Transcription and diarization complete.")

2026-07-09 02:02:06 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-07-09 02:02:06 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`


2026-07-09 02:02:06 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-3.1


config.yaml:   0%|          | 0.00/469 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/26.6M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(


2026-07-09 02:02:20 - whisperx.asr - INFO - Detected language: en (1.00) in first 30s of audio
Downloading: "https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth" to /root/.cache/torch/hub/checkpoints/wav2vec2_fairseq_base_ls960_asr_ls960.pth


100%|██████████| 360M/360M [00:01<00:00, 286MB/s]
/usr/local/lib/python3.12/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)


2026-07-09 02:02:56 - whisperx.asr - INFO - Detected language: en (1.00) in first 30s of audio
2026-07-09 02:03:27 - whisperx.asr - INFO - Detected language: en (1.00) in first 30s of audio
Batch processing complete.


In [ ]:
!cat vllm_server.log

In [ ]:
from pathlib import Path
from pyannote.database.util import load_rttm
from pyannote.metrics.diarization import DiarizationErrorRate
import pandas as pd

GT_DIR = Path("/content/evaluation_dataset/ground_truth")
PRED_DIR = Path("/content/evaluation_dataset/predictions")
der_metric = DiarizationErrorRate()
results = []

for gt_path in GT_DIR.glob("*.rttm"):
    file_id = gt_path.stem
    pred_path = PRED_DIR / f"{file_id}.rttm"
    if not pred_path.exists(): continue

    ref = load_rttm(str(gt_path)).get(file_id)
    hyp = load_rttm(str(pred_path)).get(file_id)
    if ref is None or hyp is None: continue

    file_der = der_metric(ref, hyp)
    results.append({
"File ID": file_id,
        # Diarization Error Rate: Total error percentage (False Alarm + Missed Detection + Confusion)
        "DER (%)": round(file_der * 100, 2),

        # % of time the model falsely detected speech during silence or background noise
        "False Alarm (%)": round(der_metric['false alarm'] / der_metric['total'] * 100, 2) if der_metric['total'] > 0 else 0,

        # % of time the model completely missed actual human speech
        "Missed Detection (%)": round(der_metric['missed detection'] / der_metric['total'] * 100, 2) if der_metric['total'] > 0 else 0,

        # % of time the model assigned the detected speech to the wrong speaker
        "Confusion (%)": round(der_metric['confusion'] / der_metric['total'] * 100, 2) if der_metric['total'] > 0 else 0

    })

print(pd.DataFrame(results).to_string(index=False))
print(f"GLOBAL DER: {abs(der_metric):.2%}")

/usr/local/lib/python3.12/dist-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(


Evaluating Predictions against Ground Truth...

       File ID  DER (%)  False Alarm (%)  Missed Detection (%)  Confusion (%)
callhome_eng_0    26.56             8.68                 14.17           3.71
callhome_eng_2    38.22             8.82                 13.57           9.50
callhome_eng_1    31.09             5.00                 19.47           6.97
--------------------------------------------------
GLOBAL DATASET DER: 31.44%
--------------------------------------------------


In [ ]:
import json
from pathlib import Path
from vllm import LLM, SamplingParams

llm = LLM(
    model="Qwen/Qwen2.5-32B-Instruct-AWQ",
    quantization="AWQ",
    dtype="half",
    gpu_memory_utilization=0.9,
    max_model_len=16384
)
sampling_params = SamplingParams(temperature=0.1, max_tokens=1024)

transcripts = []
for json_file in Path("/content/").glob("*.json"):
    if json_file.name == "analysis_results.json": continue
    with open(json_file, 'r', encoding='utf-8') as f:
        try:
            transcripts.append(json.load(f))
        except json.JSONDecodeError:
            pass

print(f"Loaded {len(transcripts)} transcripts.")

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

INFO 07-09 02:42:39 awq_marlin.py:93] Detected that the model can run with awq_marlin, however you specified quantization=awq explicitly, so forcing awq. Use quantization=awq_marlin for faster inference
WARNING 07-09 02:42:39 config.py:335] awq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 07-09 02:42:39 llm_engine.py:223] Initializing an LLM engine (v0.6.1.post2) with config: model='Qwen/Qwen2.5-32B-Instruct-AWQ', speculative_config=None, tokenizer='Qwen/Qwen2.5-32B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=16384, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=awq, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, dec

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00003-of-00005.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/3.48G [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]


INFO 07-09 02:43:36 model_runner.py:1008] Loading model weights took 18.1434 GB
INFO 07-09 02:43:41 gpu_executor.py:122] # GPU blocks: 12569, # CPU blocks: 1024
INFO 07-09 02:43:43 model_runner.py:1311] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 07-09 02:43:43 model_runner.py:1315] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 07-09 02:44:26 model_runner.py:1430] Graph capturing finished in 43 secs.
Loaded 3 transcripts. Model initialized with 16k context window.


In [ ]:
import json
from google.colab import files

results = []
system_prompt = """
You are a QA Auditor for a call center. Analyze the provided transcript.
Output ONLY a valid JSON object. Keys must be in English. Values must be in Russian.
Schema:
{
    "greeting_present": true/false,
    "politeness_score": 1-10,
    "problem_resolved": true/false,
    "critical_errors_found": "Describe issues or 'None'",
    "summary": "Brief call summary"
}
"""

for item in transcripts:
    context = "\n".join([f"{seg['speaker']}: {seg['text']}" for seg in item['segments']])
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Transcript:\n{context}"}
    ]

    outputs = llm.chat(messages=messages, sampling_params=sampling_params, use_tqdm=False)
    content = outputs[0].outputs[0].text

    try:
        if "```json" in content:
            content = content.split("```json")[1].split("```")[0]
        results.append({
            "file_id": item.get("file_id", "unknown"),
            "analysis": json.loads(content)
        })
    except Exception:
        pass

output_file = "/content/analysis_results.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

files.download(output_file)

Analysis complete. Downloading results...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>